In [45]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [46]:
with open('data.txt','r+') as file:
    data = file.read()

In [47]:
import os
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", 
    temperature=0,
    api_key=os.environ["GEMINI_API_KEY"]
    )

In [48]:
template = '''
You are a helpful assistant that can answer questions about the product.
Answer the question based on the product details provided.
product : {product}
question : {question}
'''

prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['product', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant that can answer questions about the product.\nAnswer the question based on the product details provided.\nproduct : {product}\nquestion : {question}\n')

In [49]:
import os
class Product(BaseModel):
    name:str = Field(description="Name of the product")
    price:str  = Field(description="Price of the product")
    description:str = Field(description="Description of the product")
    category:str = Field(description="Category of the product")
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", 
    temperature=0,
    api_key=os.environ["GEMINI_API_KEY"]
    ).with_structured_output(Product)

In [50]:
chain = prompt_template | model

In [51]:
user_input = [
    {
    "product": data,
    "question": "What is the price of the iPhone 15 Pro Max?"
},
{
    "product": data,
    "question": "What is the price of the Galaxy S24 Ultra?"
},
{
    "product": data,
    "question": "What is the price of the Redmi?"
},
{
    "product": data,
    "question": "What is the price of the OnePlus?"
}
]

In [52]:
prompt_template = PromptTemplate.from_template(template)
chain = prompt_template | model

In [53]:
u_input = {"product":data,'question':"What is the price of One plus 13?"}
response = chain.invoke(u_input)
response_dict = response.model_dump_json()
print(response_dict)

{"name":"OnePlus 13","price":"Not Available","description":"The product 'OnePlus 13' is not available in the provided product list. The closest available model is the OnePlus 12R.","category":"Smartphones"}


In [54]:
print(response)

name='OnePlus 13' price='Not Available' description="The product 'OnePlus 13' is not available in the provided product list. The closest available model is the OnePlus 12R." category='Smartphones'


In [55]:
response

Product(name='OnePlus 13', price='Not Available', description="The product 'OnePlus 13' is not available in the provided product list. The closest available model is the OnePlus 12R.", category='Smartphones')

In [56]:
batch_response = chain.batch(user_input)


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 36.3932488s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTie

In [ ]:
results = []
for result in  batch_response:
    results.append(result.model_dump_json())
results

['{"name":"Apple iPhone 15 Pro Max","price":"159999.00","description":"Apple iPhone 15 Pro Max featuring a 6.7 inch Super Retina XDR display, A17 Pro processor, 8GB RAM, and 256GB storage. Released in 2023. Currently out of stock.","category":"Smartphones"}',
 '{"name":"Samsung Galaxy S24 Ultra","price":"129999.99","description":"A premium smartphone featuring a 6.8 inch AMOLED display, Snapdragon 8 Gen 3 processor, 12GB RAM, and 512GB storage.","category":"Smartphones"}',
 '{"name":"Redmi Note 13 Pro+","price":"29999.99","description":"Redmi Note 13 Pro+ featuring 8GB RAM, 256GB storage, 5000mAh battery, 6.67 inch AMOLED display, and Dimensity 7200 Ultra processor.","category":"Smartphones"}',
 '{"name":"OnePlus 12R","price":"45999.00","description":"The OnePlus 12R features 16GB RAM, 256GB storage, a 5500mAh battery, and a Snapdragon 8 Gen 2 processor.","category":"Smartphones"}']

In [ ]:
batch_response

[Product(name='Apple iPhone 15 Pro Max', price='159999.00', description='Apple iPhone 15 Pro Max featuring a 6.7 inch Super Retina XDR display, A17 Pro processor, 8GB RAM, and 256GB storage. Released in 2023. Currently out of stock.', category='Smartphones'),
 Product(name='Samsung Galaxy S24 Ultra', price='129999.99', description='A premium smartphone featuring a 6.8 inch AMOLED display, Snapdragon 8 Gen 3 processor, 12GB RAM, and 512GB storage.', category='Smartphones'),
 Product(name='Redmi Note 13 Pro+', price='29999.99', description='Redmi Note 13 Pro+ featuring 8GB RAM, 256GB storage, 5000mAh battery, 6.67 inch AMOLED display, and Dimensity 7200 Ultra processor.', category='Smartphones'),
 Product(name='OnePlus 12R', price='45999.00', description='The OnePlus 12R features 16GB RAM, 256GB storage, a 5500mAh battery, and a Snapdragon 8 Gen 2 processor.', category='Smartphones')]

In [ ]:
for prompt in user_input:
    response = chain.invoke(prompt)
    print(response.model_dump())

{'name': 'iPhone 15 Pro Max', 'price': '159999.00', 'description': 'Apple iPhone 15 Pro Max featuring a 6.7 inch Super Retina XDR display, A17 Pro processor, 8GB RAM, and 256GB storage.', 'category': 'Smartphones'}


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.5-flash\nPlease retry in 31.634264497s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fre